### Assignment 6: Tracking a High-Speed Train under Biased Process Dynamics

##### Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
import pandas as pd


def generate_x_true_trajectory(num_points, a_variance, x0, v0, time_step):
    '''
    Trajectory generator: 
    x[i] = x[i-1] + v[i-1]*T + 0.5 * a[i]*T^2
    '''
    accel = np.random.normal(0, np.sqrt(a_variance), num_points)

    v = np.zeros(num_points)
    v[0] = v0
    x = np.zeros(num_points)
    x[0] = x0

    for i in range(1, num_points):
        v[i] = v[i - 1] + accel[i] * time_step
        x[i] = x[i - 1] + v[i - 1] * time_step + 0.5 * accel[i] * time_step**2

    return x, v, accel

def generate_measurements(true_trajectory, noise_variance):
    """
    Generate measurements using random walk model
    """
    num = len(true_trajectory)
    z = np.zeros(num)
    
    n = np.random.normal(0, np.sqrt(noise_variance), num)    
    
    for i in range(num):
        z[i] = true_trajectory[i] + n[i]
    
    return z, n

def create_kalman_F(time_step):
    """
    Create transition matrix F (Φ):
    [1, time_step]
    [0, 1]
    """
    F = np.array([[1, time_step],
                  [0, 1]])
    return F

def create_kalman_G(time_step):
    """
    Create input matrix G:
    [time_step**2/2]
    [time_step]
    """
    G = np.array([[0.5 * time_step**2],
                  [time_step]])
    return G

def create_kalman_H():
    """
    Create observation matrix H:
    [1, 0]
    """
    H = np.array([[1, 0]])
    return H

def create_kalman_Q(a_variance, time_step):
    """
    Create process noise covariance matrix Q
    Q = G * G^T * sigma_a^2 
    """
    G = create_kalman_G(time_step)
    Q = G @ G.T * a_variance
    return Q

def create_kalman_R(measurement_variance):
    """
    Create measurement noise covariance matrix R
    """
    R = np.array([[measurement_variance]])
    return R

def kalman_filter_predict(X_prev, P_prev, F, Q):
    """
    Prediction
    X_{i,i-1} = F * X_{i-1,i-1}
    P_{i,i-1} = F * P_{i-1,i-1} * F^T + Q
    """
    X_pred = F @ X_prev
    P_pred = F @ P_prev @ F.T + Q
    return X_pred, P_pred

def kalman_filter_update(X_pred, P_pred, z, H, R):
    """
    Update step of Kalman filter
    """
    # Innovation (residual)
    r_i = z - H @ X_pred
    
    # Innovation covariance
    S = H @ P_pred @ H.T + R
    
    # Kalman gain
    K = P_pred @ H.T @ np.linalg.inv(S)
    
    # Update estimate
    X_update = X_pred + K @ r_i
    
    # Update covariance
    P_update = (np.eye(2) - K @ H) @ P_pred # np.eye gives us[[1 0] [0 1]](I)
    
    return X_update, P_update, K

def run_kalman_filter(z_measurements, time_step, a_variance, measurement_variance, 
                      X_init, P_init, num_points):
    """
    Complete Kalman filter implementation
    """
    
    F = create_kalman_F(time_step)
    H = create_kalman_H()
    Q = create_kalman_Q(a_variance, time_step)
    R = create_kalman_R(measurement_variance)
    
    X_filtered = np.zeros((num_points, 2))      
    P_filtered = np.zeros((num_points, 2, 2))   
    X_predicted = np.zeros((num_points, 2))     
    P_predicted = np.zeros((num_points, 2, 2))  
    K_gains = np.zeros((num_points, 2, 1))      
    
    X_filtered[0] = X_init.flatten()
    P_filtered[0] = P_init
    
    for i in range(1, num_points):
        # Prediction step
        X_pred, P_pred = kalman_filter_predict(
            X_filtered[i-1].reshape(-1,1), P_filtered[i-1], F, Q)
        X_predicted[i] = X_pred.flatten()
        P_predicted[i] = P_pred
        
        # Update step
        X_update, P_update, K = kalman_filter_update(
            X_pred, P_pred, z_measurements[i], H, R)
        X_filtered[i] = X_update.flatten()
        P_filtered[i] = P_update
        K_gains[i] = K
    
    return X_filtered, P_filtered, X_predicted, P_predicted, K_gains

def run_kalman_filter_with_bias(z_measurements, time_step, a_variance, measurement_variance, 
                               X_init, P_init, num_points, bias_q):
    """
    Kalman filter implementation with bias correction
    """
    
    F = create_kalman_F(time_step)
    H = create_kalman_H()
    Q = create_kalman_Q(a_variance, time_step)
    R = create_kalman_R(measurement_variance)
    G = create_kalman_G(time_step)
    
    X_filtered = np.zeros((num_points, 2))      
    P_filtered = np.zeros((num_points, 2, 2))   
    X_predicted = np.zeros((num_points, 2))     
    P_predicted = np.zeros((num_points, 2, 2))  
    K_gains = np.zeros((num_points, 2, 1))      
    
    X_filtered[0] = X_init.flatten()
    P_filtered[0] = P_init
    
    for i in range(1, num_points):
        # Prediction step with bias
        X_pred = F @ X_filtered[i-1].reshape(-1,1) + G * bias_q
        P_pred = F @ P_filtered[i-1] @ F.T + Q
        X_predicted[i] = X_pred.flatten()
        P_predicted[i] = P_pred
        
        # Update step
        X_update, P_update, K = kalman_filter_update(
            X_pred, P_pred, z_measurements[i], H, R)
        X_filtered[i] = X_update.flatten()
        P_filtered[i] = P_update
        K_gains[i] = K
    
    return X_filtered, P_filtered, X_predicted, P_predicted, K_gains

def calculate_residuals(z_measurements, X_predicted, H):
    """
    Calculate filter residuals
    """
    num_points = len(z_measurements)
    residuals = np.zeros(num_points)
    for i in range(num_points):
        residual_value = z_measurements[i] - (H @ X_predicted[i].reshape(-1,1))[0, 0]
        residuals[i] = residual_value
    return residuals

def calculate_residual_variance(P_predicted, H, R):
    """
    Calculate variance of residuals S_i
    """
    num_points = len(P_predicted)
    S = np.zeros(num_points)
    for i in range(num_points):
        S_value = (H @ P_predicted[i] @ H.T + R)[0, 0]
        S[i] = S_value
    return S

def calculate_systematic_error(time_step, num_points, bias_q, K_gains, H):
    """
    Calculate systematic error propagation
    """
    F = create_kalman_F(time_step)
    G = create_kalman_G(time_step)
    
    mu_pred = np.zeros((num_points, 2))  # μ{i,i-1}
    mu_filt = np.zeros((num_points, 2))  # μ{i,i}
    
    # Initial condition: μ{0,0} = [0, 0]
    mu_filt[0] = [0, 0]
    
    for i in range(1, num_points):
        # Prediction step: μ{i,i-1} = Φ * μ{i-1,i-1} + G * q
        mu_pred[i] = F @ mu_filt[i-1] + G.flatten() * bias_q
        
        # Filtration step: μ{i,i} = (I - K_i * H) * μ{i,i-1}
        K = K_gains[i].flatten()
        I_KH = np.eye(2) - K.reshape(-1,1) @ H
        mu_filt[i] = I_KH @ mu_pred[i]
    
    return mu_pred, mu_filt

In [ ]:
fontsize_title = 30
fontsize_label = 20
fontsize_leg = 20

In [ ]:
#Reading measurements of position in meters, provided by GPS with 2 seconds interval
gps_data = np.loadtxt('GPS.txt')

In [ ]:
# Parameters
num_points = len(gps_data)
T = 2  
sigma_a2 = 0.2**2  
sigma_eta2 = 20**2  

# Initial conditions
X_init = np.array([[2], [0]])  
P_init = np.array([[100, 0], [0, 100]])  

#### Step I. Filtering the data

In [ ]:
# Run Kalman filter
X_filtered, P_filtered, X_predicted, P_predicted, K_gains = run_kalman_filter(gps_data, T, sigma_a2, sigma_eta2, X_init, P_init, num_points)

# Extract position and velocity estimates
x_estimates = X_filtered[:, 0]
v_estimates = X_filtered[:, 1]

time_axis = np.arange(num_points) * T

In [ ]:
# Plot: GPS measurements of the position 𝑥, filter estimates for both the coordinate 𝑥 and velocity
plt.figure(figsize=(20, 10))

plt.subplot(2, 1, 1)

plt.plot(time_axis, gps_data, 'b.-', markersize=4, label='GPS measurements')
plt.plot(time_axis, x_estimates, 'r-', linewidth=1, label='Filter position estimate')

plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.title('\n Fig.1.1. GPS measurements and position estimate \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.subplot(2, 1, 2)

plt.plot(time_axis, gps_data, 'b.-', markersize=4, label='GPS measurements')
plt.plot(time_axis, x_estimates, 'r-', linewidth=1, label='Filter position estimate')

plt.ylabel('\n Position, m \n', fontsize=fontsize_label)
plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.title('\n Fig.1.2. GPS measurements and position estimate zoomed \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.xlim([10, 45])
plt.ylim([0, 250])


plt.figure(figsize=(20, 7))

plt.plot(time_axis, v_estimates, 'g-', linewidth=2, label='Velocity estimate')

plt.ylabel('\n Velocity, m/s \n', fontsize=fontsize_label)
plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.title('\n Fig.2. Velocity estimate \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.tight_layout()
plt.show()

The filtered position estimate may seen slightly overconfident, indicated by the near-perfect overlap, but if we zooms it, we can see shift, the difference between the measurements that are above the filtered ones is greater than the difference between the measurements that are below. The position plot appears smooth because it's directly measured by GPS and filtered, while the velocity plot shows more noise since velocity is indirectly estimated through the Kalman filter's.

#### Step II. Checking residuals for signs of bias

In [ ]:
H = create_kalman_H()
R = create_kalman_R(sigma_eta2)

# Calculate residuals and their variance
residuals = calculate_residuals(gps_data, X_predicted, H)
S = calculate_residual_variance(P_predicted, H, R)
std_residuals = np.sqrt(S)

In [ ]:
mean_residual = np.mean(residuals)
print(f"Mean residual: {mean_residual:.3f} m")

In [ ]:
# Plot residuals
plt.figure(figsize=(20, 10))

plt.plot(time_axis, residuals, 'b-', linewidth=1, label='Filter residuals')
plt.plot(time_axis, std_residuals, 'r--', linewidth=2, label='+√S_i')
plt.plot(time_axis, -std_residuals, 'r--', linewidth=2, label='-√S_i')
plt.fill_between(time_axis, std_residuals, -std_residuals, alpha=0.2, color='red')

plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.ylabel('\n Residual, m \n', fontsize=fontsize_label)
plt.title('\n Fig.3. Filter residuals \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

The mean of residual is equal to 20.861 m, so we can not say that is close to zero. This does indicate a potential bias. An optimal Kalman filter should have zero mean.
Residuals frequently exceed +√𝑆i limits, it suggests that the model or measurements are biased. 

#### Step III. Correcting the bias

In [ ]:
bias_q = 0.2  # Known bias value

# Run Kalman filter with bias correction
X_filtered_bias, P_filtered_bias, X_predicted_bias, P_predicted_bias, K_gains_bias = run_kalman_filter_with_bias(gps_data, T, sigma_a2, sigma_eta2, X_init, P_init, num_points, bias_q)

# Calculate residuals after bias correction
residuals_bias = calculate_residuals(gps_data, X_predicted_bias, H)
S_bias = calculate_residual_variance(P_predicted_bias, H, R)
std_residuals_bias = np.sqrt(S_bias)

In [ ]:
mean_bias_residual = np.mean(residuals_bias)
print(f"Mean bias residual: {mean_bias_residual:.3f} m")

In [ ]:
# Plot comparison of residuals before and after correction
plt.figure(figsize=(20, 10))

plt.subplot(2, 1, 1)

plt.plot(time_axis, residuals, 'b-', linewidth=1, label='Residuals (no correction)')
plt.plot(time_axis, std_residuals, 'r--', linewidth=2, label='±√S_i (no correction)')
plt.plot(time_axis, -std_residuals, 'r--', linewidth=2)
plt.fill_between(time_axis, std_residuals, -std_residuals, alpha=0.2, color='red')

plt.ylabel('\n Residual, m \n', fontsize=fontsize_label)
plt.title('\n Fig.4. Residual comparison before and after bias correction \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.subplot(2, 1, 2)

plt.plot(time_axis, residuals_bias, 'g-', linewidth=1, label='Residuals (with correction)')
plt.plot(time_axis, std_residuals_bias, 'm--', linewidth=2, label='±√S_i (with correction)')
plt.plot(time_axis, -std_residuals_bias, 'm--', linewidth=2)
plt.fill_between(time_axis, std_residuals_bias, -std_residuals_bias, alpha=0.2, color='magenta')

plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.ylabel('\n Residual, m \n', fontsize=fontsize_label)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.tight_layout()
plt.show()

The mean of residual is equal to -0.959 m, it is close to zero. The filter is not systematically shifted in one direction: on average,
measurements agree with predictions. Majority of residuals falls within the corridor ±√𝑆i, the filter’s estimation (𝑆𝑖) matches reality (𝑟𝑖) quite well. The errors behave as the filter expects, and the tracking remains stable. Now the filter keeps well the train “on track”

#### Step IV: Analyzing the true error when ignoring bias of state noise

In [ ]:
# Calculate systematic error for filter without bias correction
mu_pred, mu_filt = calculate_systematic_error(T, num_points, bias_q, K_gains, H)

# Calculate true total error (random + systematic)
true_error_x = np.zeros(num_points)
true_error_v = np.zeros(num_points)
filter_error_x = np.zeros(num_points)
filter_error_v = np.zeros(num_points)

for i in range(num_points):
    # True error = random error (from P) + systematic error (from μ)
    true_error_x[i] = np.sqrt(P_filtered[i, 0, 0] + mu_filt[i, 0]**2)
    true_error_v[i] = np.sqrt(P_filtered[i, 1, 1] + mu_filt[i, 1]**2)
    
    # Filter error estimate (only random component)
    filter_error_x[i] = np.sqrt(P_filtered[i, 0, 0])
    filter_error_v[i] = np.sqrt(P_filtered[i, 1, 1])

In [ ]:
# Plot error comparison
plt.figure(figsize=(20, 10))

plt.subplot(2, 1, 1)

plt.plot(time_axis, true_error_x, 'r-', linewidth=2, label='True position error')
plt.plot(time_axis, filter_error_x, 'b--', linewidth=2, label='Filter position error estimate')
plt.plot(time_axis, np.ones(num_points) * np.sqrt(sigma_eta2), 'g-', linewidth=1, label='The standard deviation of the measurement noise')

plt.ylabel('\n Error, m \n', fontsize=fontsize_label)
plt.title('\n Fig.5. Error comparison \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.subplot(2, 1, 2)

plt.plot(time_axis, true_error_v, 'r-', linewidth=2, label='True velocity error')
plt.plot(time_axis, filter_error_v, 'b--', linewidth=2, label='Filter velocity error estimate')

plt.xlabel('\n Time, s \n', fontsize=fontsize_label)
plt.ylabel('\n Error, m/s \n', fontsize=fontsize_label)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)

plt.tight_layout()
plt.show()



The filter's calculated covariance matrices 𝑃 only represent the random component of the error, but ignore the systematic bias component. The measurement noise standard deviation is 20 m. The true total position error (which includes the systematic bias) stabilizes at 20 m. The standard deviation from the filter settles around 10 m. That mismatch means P is under-estimating total uncertainty by roughly a factor of 2 for position in this case.

The true velocity error stabilizes around 3 m/s. The filter’s predicted velocity uncertainty stabilizes around 1 m/s. P underestimates the total velocity error by roughly a factor of 3 in this case.

The filter’s internal estimates are overly optimistic: while the filter assumes it is performing effectively, the true error behavior demonstrates the opposite — the estimation accuracy is degraded due to the neglected process bias. 

#### Conclusion
During this assignment, we:

- ran Kalman filter on the provided train GPS trajectory data.  The filtered position estimate closely followed the GPS data. But, with zooming we see a downward shift in our graph relative to the measurements. The velocity estimates appeared noisier, as velocity is derived indirectly from position updates.
- computed filter residuals and their expected standard deviation. The mean residual is 20.861 m, far from zero, indicating a potential systematic bias. Residuals frequently fell outside the bounds ±√𝑆, confirming that the filter’s uncertainty model was inconsistent with reality due to unmodeled bias.
- ran Kalman filter with modify the prediction step of the filter to include known bias term at the prediction step 𝑞 = 0.2. The mean residual dropped to -0.959 m, close to zero. Most residuals now stayed within the ±√𝑆 corridor, indicating that the filter was well-calibrated and the bias was effectively corrected. 
- compared the resulting true error with the standard deviation from the filter and with the standard deviation of the measurement noise. When bias is ignored, the filter underestimates its error(by of 2 for position and 3 for velocity). 